In [1]:
#easier to use notebook when getting started: using this for minor bits as I expand the database
# Krista 28 August 2026

In [1]:
#there is still some cache in Jupyter notebook that is causing issues

%load_ext autoreload
%autoreload 2 #the 2 means be aggressive and always reload imported modules

In [2]:
#will need to run this if I intend to re run populate_db.py, otherwise Jupyter notebook keeps the file open
#and the delete step in that *py file fails, KL 3 September 2026
def closeDatabaseNotebook():
    import gc
    import sys

    session.close()
    engine.dispose()

    # Clear Jupyter's hidden historical output cache (which holds old results)
    sys.modules[__name__].__dict__.pop('_', None)
    sys.modules[__name__].__dict__.pop('__', None)

    # Force immediate garbage collection to release the file lock
    gc.collect()
    
    
closeDatabaseNotebook()    

NameError: name 'session' is not defined

In [8]:
from sqlalchemy import select, func, create_engine, text
from sqlalchemy.orm import sessionmaker, Session
from sqlalchemy import inspect, MetaData, Table
import pandas as pd

import pdb

import models 
#from models import *
# import models as m

In [4]:
#Make sure I think I have what I need:
for mapper in models.Base.registry.mappers:
    print(mapper.class_.__name__)

SeqBasics
NCBIinhouse
Discrete
LTTs1
SeqV1V2
MtabUntargetedInfo
NCBIunreleased
MetaboliteInfo
SeqV4_16S
CyverseInfo
SeqV4_18S
LTTdeep
NCBIonline


In [5]:
# create a SQLite database engine
SQLALCHEMY_DATABASE_URL = "sqlite:///test_data/sargasso.db"
#this will end up creating a new database everytime, but I need this for testing right now
#SQLALCHEMY_DATABASE_URL = f"sqlite:///test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL) #, echo=True) #(turn off echo, gets annoying)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

In [6]:
from sqlalchemy import create_engine
from sqlalchemy.ext.automap import automap_base

# 1. Reset the engine connection pool
if 'engine' in globals():
    engine.dispose()
engine = create_engine('sqlite:///test_data/sargasso.db')

# 2. CRITICAL: Initialize a completely NEW Base object to drop old classes
Base = automap_base()

# 3. Reflect from scratch
Base.prepare(autoload_with=engine)

# Your updated classes will now be freshly populated
print(Base.classes.keys()) 

['LTTs1', 'cyverse', 'seqBasics', 'discrete', 'metabolitesUntargeted', 'seqV4_16S', 'seqV1V2', 'metabolites', 'seqV4_18S', 'NCBIinhouse', 'LTTdeep', 'NCBIonline', 'NCBIunreleased']


In [9]:
# Define the SQL command to create the table and populate it simultaneously
query = (
    select(models.NCBIonline, models.NCBIinhouse)
    .join(
        models.NCBIonline,
        models.NCBIonline.biosample == models.NCBIinhouse.biosample
    )
)

df = pd.read_sql_query(query, con=engine)

# # Execute the command
# with engine.begin() as connection:
#     connection.execute(ctas_query)
#     print("🚀 New table 'CombinedSeqInfo' successfully created and populated!")

In [10]:
df.head()

,id,biosample,sample,id_1,biosample_1,cruise5,sampleV1V2,sraV1V2,seqV1V2,sampleV416s,sraV416s,seqV416s,firstReference,bottleID
0,549,SAMN28811500,1032100301_16S_V1V2,1,SAMN28811500,10321,10321_1,SRR19520215,lane1-s039-indexN703-C-S516-C-AGGCAGAA-CCTAGAG...,10321_1_S6,SRR31627557,lane1_s006_index_CGAGAGTT_CGTGAGTG_10321_1_S6,"Bolanos et al., 2022",1032100301
1,233,SAMN44822323,10321_40_S7,2,SAMN44822323,10321,10321_40,None,lane1-s040-indexN703-C-S517-C-AGGCAGAA-GCGTAAG...,10321_40_S7,SRR31399359,lane1_s007_index_CGAGAGTT_GGATATCT_10321_40_S7,None,1032100304
2,232,SAMN44822324,10321_80_S8,3,SAMN44822324,10321,10321_80,None,lane1-s041-indexN703-C-S518-C-AGGCAGAA-CTATTAA...,10321_80_S8,SRR31399358,lane1_s008_index_CGAGAGTT_GACACCGT_10321_80_S8,None,1032100306
3,231,SAMN44822325,0321_120_S9,4,SAMN44822325,10321,10321_120,None,lane1-s042-indexN703-C-S520-C-AGGCAGAA-AAGGCTA...,0321_120_S9,SRR31399158,lane1_s009_index_GACATAGT_ATCGTACG_10321_120_S9,None,1032100308
4,230,SAMN44822326,10321_160_S10,5,SAMN44822326,10321,10321_160,None,lane1-s043-indexN703-C-S521-C-AGGCAGAA-GAGCCTT...,10321_160_S10,SRR31399275,lane1_s010_index_GACATAGT_ACTATCTG_10321_160_S10,None,1032100310


SeqInfoNCBIinhouse
SeqInfoBasics
CyverseInfo
SeqInfoNCBIonline
SeqInfoLTTs1
MtabUntargetedInfo
SeqInfoV4_18S
SeqInfoV1V2
SeqInfoLTTdeep
NCBIunreleased
DiscreteInfo
MetaboliteInfo
SeqInfoV4_16S


In [48]:
#reflect the tables so I can work on them
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)

test = Table('SeqInfoNCBIinhouse', metadata_obj, autoload_with=engine)
#test = Table('SeqInfoNCBIonline', metadata_obj, autoload_with=engine)
#test = Table('SeqInfoNCBIonline', metadata_obj, autoload_with=engine)

session.query(test).all()[:5] #list so head will not work
#models.SeqInfoNCBIonline() #not helpful, just gives me the columns I defined to show in models.py

[(1, 'SAMN28811500', '10321', '10321_1', 'SRR19520215', 'lane1-s039-indexN703-C-S516-C-AGGCAGAA-CCTAGAGT-10321-1_S39', '10321_1_S6', 'SRR31627557', 'lane1_s006_index_CGAGAGTT_CGTGAGTG_10321_1_S6', 'Bolanos et al., 2022', '1032100301'),
 (2, 'SAMN44822323', '10321', '10321_40', None, 'lane1-s040-indexN703-C-S517-C-AGGCAGAA-GCGTAAGA-10321-2_S40', '10321_40_S7', 'SRR31399359', 'lane1_s007_index_CGAGAGTT_GGATATCT_10321_40_S7', None, '1032100304'),
 (3, 'SAMN44822324', '10321', '10321_80', None, 'lane1-s041-indexN703-C-S518-C-AGGCAGAA-CTATTAAG-10321-3_S41', '10321_80_S8', 'SRR31399358', 'lane1_s008_index_CGAGAGTT_GACACCGT_10321_80_S8', None, '1032100306'),
 (4, 'SAMN44822325', '10321', '10321_120', None, 'lane1-s042-indexN703-C-S520-C-AGGCAGAA-AAGGCTAT-10321-4_S42', '0321_120_S9', 'SRR31399158', 'lane1_s009_index_GACATAGT_ATCGTACG_10321_120_S9', None, '1032100308'),
 (5, 'SAMN44822326', '10321', '10321_160', None, 'lane1-s043-indexN703-C-S521-C-AGGCAGAA-GAGCCTTA-10321-5_S43', '10321_160_S10

In [49]:
query = (
    select(models.SeqInfoNCBIinhouse)
    .where(models.SeqInfoNCBIinhouse.biosample.not_like('SAMN%'))
)

# Execute the query
results = session.execute(query).scalars().all()
results

[SeqInfoNCBIonline(id=95, cruise5='AE1614'),
 SeqInfoNCBIonline(id=96, cruise5='AE1614'),
 SeqInfoNCBIonline(id=97, cruise5='AE1614'),
 SeqInfoNCBIonline(id=98, cruise5='AE1614'),
 SeqInfoNCBIonline(id=99, cruise5='AE1614'),
 SeqInfoNCBIonline(id=100, cruise5='AE1614'),
 SeqInfoNCBIonline(id=101, cruise5='AE1614'),
 SeqInfoNCBIonline(id=102, cruise5='AE1614')]

In [50]:
query = (
    select(models.SeqInfoNCBIinhouse)
    .where(models.SeqInfoNCBIinhouse.bottleID == 9161400401)
)

# Execute the query
results = session.execute(query).scalars().all()
results

[SeqInfoNCBIonline(id=95, cruise5='AE1614')]

In [51]:
query = (
    select(models.SeqInfoNCBIonline, models.SeqInfoNCBIinhouse)
    .join(
        models.SeqInfoNCBIonline,
        models.SeqInfoNCBIonline.biosample == models.SeqInfoNCBIinhouse.biosample)
    )

In [52]:
matches = session.execute(query).all()

In [53]:
matches[:5]

[(SeqInfoNCBIonline(id=549, sample='1032100301_16S_V1V2', biosample='SAMN28811500'), SeqInfoNCBIonline(id=1, cruise5='10321')),
 (SeqInfoNCBIonline(id=233, sample='10321_40_S7', biosample='SAMN44822323'), SeqInfoNCBIonline(id=2, cruise5='10321')),
 (SeqInfoNCBIonline(id=232, sample='10321_80_S8', biosample='SAMN44822324'), SeqInfoNCBIonline(id=3, cruise5='10321')),
 (SeqInfoNCBIonline(id=231, sample='0321_120_S9', biosample='SAMN44822325'), SeqInfoNCBIonline(id=4, cruise5='10321')),
 (SeqInfoNCBIonline(id=230, sample='10321_160_S10', biosample='SAMN44822326'), SeqInfoNCBIonline(id=5, cruise5='10321'))]

In [20]:
query = (
    select(models.SeqInfoNCBIonline, models.SeqInfoNCBIinhouse)
    .join(
        models.SeqInfoNCBIonline,
        models.SeqInfoNCBIonline.biosample == models.SeqInfoNCBIinhouse.biosample,
        isouter=True)
    .where(models.SeqInfoNCBIinhouse.biosample==None)
    )

matches = session.execute(query).all()[:5]

In [22]:
query = (
    select(models.SeqInfoNCBIonline)
    .where(
        ~select(models.SeqInfoNCBIonline)
        .where(models.SeqInfoNCBIonline.biosample == models.SeqInfoNCBIinhouse.biosample)
        .exists()
        )
    )
matches = session.execute(query).all()[:5]

In [ ]:
one = 'SAMN52634710'
query = select(SeqInfoNCBIonline).where(SeqInfoNCBIonline.biosample == one)
result = session.execute(query).scalars().first()

if result:
    print(f"Success! Found record ID: {result.id}")
    # Because of your mapping, you can also easily access its parent metadata:
    if result.parent_ncbi:
        print(f"Associated Cruise: {result.parent_ncbi.cruise5}")
else:
    print(f"Biosample {one} was not found in the online table.")

In [ ]:
#there is only one case where a biosample is used twice, this is a duplicate sample in Nicole and Fabian's dataset
dfc = df['biosample'].value_counts().reset_index()
dfc[dfc['count']>1]

# dfc = df['New_Bottle_ID'].value_counts().reset_index()
# dfc[dfc['count']>1]

In [ ]:
#and now it is clear that there are multiple biosamples for some New_ID (18S v. V1V2 and V4_16s)
#Need to pull the NewID from samples at NCBI without that information

In [ ]:
dfc = df['New_Bottle_ID'].value_counts().reset_index()
dfc[dfc['count']>1]

In [ ]:
#there are way too many samples with temperature at the same value (e.g., 21.142, which happens 1180 !)
len(df[df['temp'] == '21.142'])

#This is clearly a problem...oddly salinity varies a little more

In [ ]:
#existing dates are yyyy-mm-dd ...but the LTT paper does not list the day (but has ID, so pull that info from NewID)
#dft['Year'].astype(str) + '-' + dft['Month'].astype(str) + '-' + dft['Day'].astype(str)
#later

In [ ]:
# Returns rows in df['A'] that are NOT found anywhere in df['B']
# df['A'][~df['A'].isin(df['B'])]

#df['New_Bottle_ID'][~df['New_Bottle_ID'].isin(dft['Sample.ID'])]

dft['Sample.ID'][~dft['Sample.ID'].isin(df['New_Bottle_ID'])]

# dft['Sample.ID'])
# setB = set(df['New_Bottle_ID'])

In [ ]:
df['New_Bottle_ID'][~df['New_Bottle_ID'].isin(dft['Sample.ID'])]

In [ ]:
df['New_Bottle_ID'].isin(dft['Sample.ID'])

In [ ]:
df['match']  = dft['Sample.ID'].isin(df['New_Bottle_ID'])

In [ ]:
df[df['match']==True]

In [ ]:
## But looking at this, something is wrong with my logic

In [ ]:
df[df['New_Bottle_ID'] == 1032501412]
#this will list two from the NCBI list in Google Drive, one of which overlaps with the LTT paper list...but the LTT paper lists another biosample, but I cannot find that anywhere:
#SAMN52634884

In [ ]:
df.loc[1010,]

In [ ]:
df[df['biosample'] == 'SAMN52634884']

In [ ]:
df[df['biosample'] == 'SAMN44822347']

In [ ]:
dft[dft['BioSample'] == dfd.loc[0,'BioSample']]

In [ ]:
df.to_excel('../test_data/out5.xlsx')

In [ ]:
#Stick some code below this spot as a holding zone
raise SystemExit("Stop execution here")

In [ ]:
df['zSearch'] = pd.to_numeric(df['depth'],errors='coerce').round(0).astype('Int64')
dp = df.pop('zSearch')
ii = df.columns.get_loc('depth') + 1
df.insert(ii,'zRound',dp)

In [ ]:
df['zSearch'] = pd.to_numeric(df['depth'],errors='coerce')
ud = df['zSearch'].round(0).astype('Int64')
ud.unique()

In [ ]:
os.getcwd()